# MSLG-SPA 2026 - Bidirectional Gloss <-> Spanish Translation

**Task:** IberLEF 2026 MSLG-SPA shared task
**Model:** mBART-large-50 + LoRA
**Metrics (official):** BLEU + TER + chrF
**System output deadline:** 2026-04-30

---
## Before you run
1. `Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`
2. Google Drive must contain:
   ```
   MyDrive/ML_projects/mslg-spa-2026/data/raw/
   |-- MSLG_SPA_train.txt          (training set, required)
   |-- external_spanish.txt        (optional, for back-translation)
   |-- test_mslg2spa.tsv           (required before running predict)
   `-- test_spa2mslg.tsv           (required before running predict)
   ```
3. Run all cells top-to-bottom. Sections 1-4 are idempotent and safe to re-run.
   Section 4 (checkpoint restore) is a no-op on first run and auto-resumes on subsequent runs.


## 1 - Environment setup


In [ ]:
# 1.1 - GPU check
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected.\n"
        "Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU"
    )

device   = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB")
print(f"torch {torch.__version__}  |  CUDA {torch.version.cuda}")


In [ ]:
# 1.2 - Mount Drive + configure paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ============================================================
#  CONFIGURE THESE PATHS - edit only here
# ============================================================
DRIVE_BASE = Path("/content/drive/MyDrive/ML_projects/mslg-spa-2026")
DRIVE_DATA = DRIVE_BASE / "data/raw"
DRIVE_CKPT = DRIVE_BASE / "checkpoints"
DRIVE_SUB  = DRIVE_BASE / "submissions"
# ============================================================

PROJECT_ROOT = Path("/content/mslg-spa-2026")
LOCAL_DATA   = Path("/content/data_local")

for d in [DRIVE_DATA, DRIVE_CKPT, DRIVE_SUB]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive base : {DRIVE_BASE}")
print(f"Drive data : {DRIVE_DATA}")
print(f"Drive ckpt : {DRIVE_CKPT}")
print(f"Drive sub  : {DRIVE_SUB}")

In [ ]:
# 1.3 - Install packages (transformers/peft stack)
# Colab has torch, numpy, pandas, sklearn, pyyaml already.
!pip install -q transformers==4.46.0 peft==0.13.2 sentencepiece==0.2.0 \
    sacrebleu==2.4.3 evaluate==0.4.3 nltk==3.9.1

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
print("Packages installed.")


## 2 - Clone project from GitHub

Always pulls the latest code from `main`. Re-run this cell at any time to sync.

In [ ]:
# 2.0 - Clone or update project from GitHub
import subprocess
from pathlib import Path

REPO_URL     = "https://github.com/marcoBorto2921/mslg-spa-2026.git"
PROJECT_ROOT = Path("/content/mslg-spa-2026")

if PROJECT_ROOT.exists():
    result = subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "pull"],
        capture_output=True, text=True
    )
else:
    result = subprocess.run(
        ["git", "clone", REPO_URL, str(PROJECT_ROOT)],
        capture_output=True, text=True
    )

print(result.stdout or result.stderr)
print(f"Project root: {PROJECT_ROOT}")


## 3 - Data setup

Copies training data from Drive into `/content/data_local/` (RAM). Test files are copied if present; otherwise the cell warns and continues.


In [ ]:
# 3.1 - Copy training + test data from Drive to local RAM
import shutil
from pathlib import Path

LOCAL_DATA = Path("/content/data_local")
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

def copy_if_exists(name, required=False):
    src = DRIVE_DATA / name
    dst = LOCAL_DATA / name
    if src.exists():
        shutil.copy2(src, dst)
        print(f"  OK      {name}")
        return True
    msg = f"  MISSING {name}"
    if required:
        raise FileNotFoundError(f"Required file not found on Drive: {src}")
    print(msg + "  (optional — skipping)")
    return False

copy_if_exists("MSLG_SPA_train.txt", required=True)
copy_if_exists("external_spanish.txt", required=False)
has_test_m2s = copy_if_exists("MSLG2SPA_test.txt", required=False)
has_test_s2m = copy_if_exists("SPA2MSLG_test.txt", required=False)

print()
if not (has_test_m2s and has_test_s2m):
    print("WARNING: Test files missing. Training will work, prediction cells will fail.")
else:
    print("All test files present.")

In [ ]:
# 3.2 - Patch config paths (point all configs to /content/data_local)
import os, sys, yaml

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

CONFIGS_TO_PATCH = [
    "configs/baseline.yaml",
    "configs/strong.yaml",
    "configs/tuned.yaml",
    "configs/tuned_bt.yaml",
]

for cfg_name in CONFIGS_TO_PATCH:
    cfg_path = PROJECT_ROOT / cfg_name
    if not cfg_path.exists():
        print(f"  SKIP    {cfg_name}  (not found)")
        continue
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    cfg["data"]["test_mslg2spa"] = str(LOCAL_DATA / "MSLG2SPA_test.txt")
    cfg["data"]["test_spa2mslg"] = str(LOCAL_DATA / "SPA2MSLG_test.txt")
    # tuned_bt uses augmented data; others use raw training file
    if "tuned_bt" not in cfg_name:
        cfg["data"]["train_file"] = str(LOCAL_DATA / "MSLG_SPA_train.txt")
    else:
        cfg["data"]["train_file"] = str(LOCAL_DATA / "augmented_train.tsv")
    with open(cfg_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
    print(f"  Patched {cfg_name}")

# Sanity check on raw training file
from src.data.dataset import load_pairs, print_stats
df = load_pairs(str(LOCAL_DATA / "MSLG_SPA_train.txt"))
print_stats(df, name="Training data")

## 4 - Restore checkpoints from Drive

Always run this cell before training. It is a no-op on first run and on subsequent runs it mirrors Drive's checkpoint directories into local. Completed subtasks are skipped automatically by the training cells.


In [ ]:
# 4.0 - Restore any existing checkpoints from Drive
import shutil
from pathlib import Path

LOCAL_CKPT_ROOT = PROJECT_ROOT / "checkpoints"
LOCAL_CKPT_ROOT.mkdir(parents=True, exist_ok=True)

copied = 0
if DRIVE_CKPT.exists():
    for item in DRIVE_CKPT.iterdir():
        target = LOCAL_CKPT_ROOT / item.name
        if target.exists():
            continue
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)
        copied += 1
        print(f"  restored  {item.name}")

if copied == 0:
    print("No checkpoints on Drive. Fresh training run.")
else:
    print(f"Restored {copied} item(s) from Drive.")


## 5 - Training

Pick a config (`baseline.yaml` for the reference chrF 52.15 run, or `strong.yaml` for the LoRA r=64 + all-linear-targets upgrade) and run both subtasks.

**Memory note** — `strong.yaml` has 34.6M trainable params. If you hit OOM on T4, lower `per_device_train_batch_size` in the config cell below to 4.


In [ ]:
# 5.1 - Choose config
# ============================================================
#  EDIT HERE
# ============================================================
CONFIG_NAME = "baseline.yaml"  # or "strong.yaml"
# ============================================================

CONFIG_PATH = f"configs/{CONFIG_NAME}"
import yaml
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
print(f"Config        : {CONFIG_PATH}")
print(f"Model         : {cfg['model']['name']}")
print(f"LoRA r        : {cfg['lora']['r']}")
print(f"LoRA targets  : {cfg['lora'].get('target_modules', '[q_proj,v_proj]')}")
print(f"Epochs        : {cfg['training']['num_train_epochs']}")
print(f"Batch         : {cfg['training']['per_device_train_batch_size']}")
print(f"Label smooth  : {cfg['training'].get('label_smoothing_factor', 0.0)}")


In [ ]:
# 5.2 - Train MSLG2SPA
# Output dir from config is relative, so checkpoints land in /content/mslg-spa-2026/<output_dir>
import os
os.chdir(str(PROJECT_ROOT))

!python scripts/train.py --config {CONFIG_PATH} --subtask mslg2spa --drive_ckpt_dir {DRIVE_CKPT}

In [ ]:
# 5.3 - Train SPA2MSLG
import os
os.chdir(str(PROJECT_ROOT))

!python scripts/train.py --config {CONFIG_PATH} --subtask spa2mslg --drive_ckpt_dir {DRIVE_CKPT}

In [ ]:
# 5.4 - Backup all checkpoints to Drive
import shutil
from pathlib import Path

LOCAL_CKPT_ROOT = PROJECT_ROOT / "checkpoints"
count = 0
for item in LOCAL_CKPT_ROOT.iterdir():
    target = DRIVE_CKPT / item.name
    if item.is_dir():
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(item, target)
    else:
        shutil.copy2(item, target)
    count += 1
    print(f"  backed up  {item.name}")

print(f"\nBacked up {count} item(s) to {DRIVE_CKPT}")


## 5.5 - EXP-004: Back-translation augmentation

Generates synthetic (MSLG, SPA) pairs using the trained SPA2MSLG model, then retrains MSLG2SPA with the augmented dataset.

**Prerequisites:** section 5 must be complete (tuned checkpoints saved to Drive).

**Expected gain:** +3–6 chrF on MSLG2SPA based on literature (4x oversample sweet spot).

Pipeline:
1. Extract 490 SPA sentences from training file + 99 external = ~589 BT source sentences
2. Translate via SPA2MSLG → synthetic glosses
3. Combine original 490 + 589 synthetic = ~1079 pairs (~2.2x)
4. Retrain MSLG2SPA with `tuned_bt.yaml`

In [ ]:
# 5.5.1 - Copy augmented data from Drive if already generated
import shutil
from pathlib import Path

LOCAL_AUG = LOCAL_DATA / "augmented_train.tsv"
DRIVE_AUG = DRIVE_DATA / "augmented_train.tsv"

if LOCAL_AUG.exists():
    print(f"augmented_train.tsv already in local RAM ({LOCAL_AUG})")
elif DRIVE_AUG.exists():
    shutil.copy2(DRIVE_AUG, LOCAL_AUG)
    print(f"Copied augmented_train.tsv from Drive ({DRIVE_AUG})")
else:
    print("augmented_train.tsv not found — will be generated in next cell")

In [ ]:
# 5.5.2 - Generate back-translation pairs
# Uses SPA2MSLG checkpoint from tuned run.
# Skip this cell if augmented_train.tsv already exists (checked in 5.5.1).
import os
from pathlib import Path

if (LOCAL_DATA / "augmented_train.tsv").exists():
    print("augmented_train.tsv exists — skipping generation")
else:
    os.chdir(str(PROJECT_ROOT))

    # ============================================================
    #  EDIT IF YOUR CHECKPOINT PATH IS DIFFERENT
    # ============================================================
    SPA2MSLG_CKPT = str(PROJECT_ROOT / "checkpoints/tuned/final")
    MSLG2SPA_CKPT = str(PROJECT_ROOT / "checkpoints/tuned/final")
    OUTPUT_TSV    = str(LOCAL_DATA / "augmented_train.tsv")
    # ============================================================

    !python scripts/back_translate.py \
        --config configs/tuned.yaml \
        --spa2mslg_checkpoint {SPA2MSLG_CKPT} \
        --mslg2spa_checkpoint {MSLG2SPA_CKPT} \
        --output {OUTPUT_TSV} \
        --extract_from_train \
        --spa_file {LOCAL_DATA}/external_spanish.txt \
        --round_trip_threshold 0.0

In [ ]:
# 5.5.3 - Save augmented data to Drive + quick stats
import shutil
from pathlib import Path
from src.data.dataset import load_pairs, print_stats

LOCAL_AUG = LOCAL_DATA / "augmented_train.tsv"
DRIVE_AUG = DRIVE_DATA / "augmented_train.tsv"

if not LOCAL_AUG.exists():
    raise FileNotFoundError("augmented_train.tsv not found. Run cell 5.5.2 first.")

shutil.copy2(LOCAL_AUG, DRIVE_AUG)
print(f"Saved to Drive: {DRIVE_AUG}")

df_aug = load_pairs(str(LOCAL_AUG))
print_stats(df_aug, name="Augmented training data")

In [ ]:
# 5.5.4 - Train MSLG2SPA with back-translated data (EXP-004)
import os
os.chdir(str(PROJECT_ROOT))

!python scripts/train.py --config configs/tuned_bt.yaml --subtask mslg2spa --drive_ckpt_dir {DRIVE_CKPT}

In [ ]:
# 5.5.5 - Backup EXP-004 checkpoints to Drive
import shutil
from pathlib import Path

LOCAL_CKPT_ROOT = PROJECT_ROOT / "checkpoints"
count = 0
for item in LOCAL_CKPT_ROOT.iterdir():
    target = DRIVE_CKPT / item.name
    if item.is_dir():
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(item, target)
    else:
        shutil.copy2(item, target)
    count += 1
    print(f"  backed up  {item.name}")

print(f"\nBacked up {count} item(s) to {DRIVE_CKPT}")

## 6 - Evaluation and submission

Runs `run_evaluate.py` if test files have references, or `predict.py` / `ensemble_predict.py` for blind submission.


In [ ]:
# 6.1 - Submission parameters
# ============================================================
#  EDIT HERE
# ============================================================
TEAM_NAME     = "mslgTeam"         # your team name
SOLUTION_NAME = "baseline_bt"       # label for this submission
N_CHECKPOINTS = 3                    # for ensemble_predict
# ============================================================
print(f"Team     : {TEAM_NAME}")
print(f"Solution : {SOLUTION_NAME}")


In [ ]:
# 6.2 - Single-model submission (MSLG2SPA)
import os
os.chdir(str(PROJECT_ROOT))
!python scripts/predict.py --config {CONFIG_PATH} --subtask mslg2spa \
    --team {TEAM_NAME} --solution {SOLUTION_NAME}


In [ ]:
# 6.3 - Single-model submission (SPA2MSLG)
import os
os.chdir(str(PROJECT_ROOT))
!python scripts/predict.py --config {CONFIG_PATH} --subtask spa2mslg \
    --team {TEAM_NAME} --solution {SOLUTION_NAME}


In [ ]:
# 6.4 - Ensemble submission (top-N checkpoints)
# Point --checkpoint_dir at the subtask-specific folder. Baseline writes to
# checkpoints/baseline and strong writes to checkpoints/strong — both are shared
# between subtasks unless the config was split. If a per-subtask subfolder exists,
# prefer that one; otherwise use the config output_dir.
import os, yaml
os.chdir(str(PROJECT_ROOT))

with open(CONFIG_PATH) as f:
    _cfg = yaml.safe_load(f)
CKPT_DIR = _cfg["training"]["output_dir"]
print(f"Checkpoint dir: {CKPT_DIR}")

!python scripts/ensemble_predict.py --config {CONFIG_PATH} --subtask mslg2spa \
    --checkpoint_dir {CKPT_DIR} --team {TEAM_NAME} --solution {SOLUTION_NAME}_ensemble{N_CHECKPOINTS} \
    --n_checkpoints {N_CHECKPOINTS}
!python scripts/ensemble_predict.py --config {CONFIG_PATH} --subtask spa2mslg \
    --checkpoint_dir {CKPT_DIR} --team {TEAM_NAME} --solution {SOLUTION_NAME}_ensemble{N_CHECKPOINTS} \
    --n_checkpoints {N_CHECKPOINTS}


In [ ]:
# 6.5 - Evaluate on val split (no test references needed) - ensemble vs single-best
# Optional sanity check before submission. Uses the same val split as training.
import os, yaml
os.chdir(str(PROJECT_ROOT))

with open(CONFIG_PATH) as f:
    _cfg = yaml.safe_load(f)
CKPT_DIR = _cfg["training"]["output_dir"]

!python scripts/ensemble_predict.py --config {CONFIG_PATH} --subtask mslg2spa \
    --checkpoint_dir {CKPT_DIR} --validate --n_checkpoints {N_CHECKPOINTS}
!python scripts/ensemble_predict.py --config {CONFIG_PATH} --subtask spa2mslg \
    --checkpoint_dir {CKPT_DIR} --validate --n_checkpoints {N_CHECKPOINTS}


In [ ]:
# 6.6 - Copy all outputs to Drive + download locally
import shutil
from pathlib import Path
from google.colab import files

outputs_dir = PROJECT_ROOT / "outputs"
if not outputs_dir.exists() or not any(outputs_dir.iterdir()):
    print("No outputs to upload.")
else:
    for f in outputs_dir.glob("*.txt"):
        dest = DRIVE_SUB / f.name
        shutil.copy2(f, dest)
        print(f"  copied to Drive: {dest}")
        files.download(str(f))
